## Cleaning Summary — orders_raw → orders_clean

Approach: work on a copy (`orders_clean`); move every bad row to a quarantine table
(`orders_quarantine`) with a reason, then remove it from the clean table. No values were
recovered (fixing them is left for later). Nothing was lost: clean + quarantine = the original 9,268.

| Step | Finding | Decision | Action | Rows |
|------|---------|----------|--------|------|
| 1 | 183 exact duplicate rows | keep one copy, remove the rest | quarantined as 'duplicate row' | 183 |
| 2 | 97 internal 'test' orders | not real sales | quarantined as 'test order' | 97 |
| 3 | 77 rows with missing category | can't trust; recovery left for later | quarantined as 'missing category' | 77 |
| 4 | 96 rows with missing customer id | can't attribute; recovery left for later | quarantined as 'missing customer id' | 96 |
| 5 | 161 rows with zero/negative quantity | not a valid order | quarantined as 'invalid quantity' | 161 |
| 6 | 13 rows priced 999999 | fake placeholder, not a real price | quarantined as 'placeholder price' | 13 |
| — | 24 rows priced 0 | plausibly free/promo items | kept (no evidence they're wrong) | 0 |

**Result:** `orders_clean` = 8,641 rows · `orders_quarantine` = 627 rows · total = 9,268 (reconciles).

**Key decisions:**
- Chose to quarantine, not delete — bad rows are preserved with a reason for later review.
- Chose not to recover missing category/customer id at this stage (business needs to decide based on quarantine).
- Kept zero-priced rows (only the obvious 999999 placeholder was removed).
- The clean table still holds text columns; conversion to real numbers/dates happens in the
  later steps that need them (EUR conversion and the revenue tables).

**Note on automation:** cleaning is a full refresh (rebuilds from raw each run) — safe and
idempotent for this small, static dataset; at scale, incremental processing would be preferred.


## 1. Set up the working table and the quarantine table

In [11]:
%%sql
-- Makes a new table called orders_clean, filled with a full copy of the raw data.
-- We clean the copy so the original orders_raw stays safe and untouched.
-- It just confirms the table was created (it now holds all 9268 rows to start).

CREATE OR REPLACE TABLE orders_clean AS
SELECT * FROM orders_raw

StatementMeta(, 225c4ce8-f213-4645-ba3b-a5c55fcbb79f, 12, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [12]:
%%sql
-- Creates an empty quarantine table with the same columns plus a "reject_reason" column.
-- This is where every bad row will be parked, tagged with the reason it was removed.
-- It confirms the table was created; it has 0 rows for now (WHERE 1 = 0 = no rows).
CREATE OR REPLACE TABLE orders_quarantine AS
SELECT *, CAST(NULL AS STRING) AS reject_reason
FROM orders_raw
WHERE 1 = 0

StatementMeta(, 225c4ce8-f213-4645-ba3b-a5c55fcbb79f, 13, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

## 2. Remove duplicate rows

In [13]:
%%sql
-- Builds a helper TABLE that numbers each copy of a row (1 = keep, 2+ = duplicate).
-- A real TABLE (not a view) so the next step can rebuild orders_clean without a self-reference.
CREATE OR REPLACE TABLE orders_ranked AS
SELECT *,
  ROW_NUMBER() OVER (
    PARTITION BY order_id, customer_id, customer_email, order_ts, status, channel, sku,
                 product_name, category, qty, unit_price, currency, country, fx_reference_date
    ORDER BY order_id
  ) AS copy_number
FROM orders_clean

StatementMeta(, 225c4ce8-f213-4645-ba3b-a5c55fcbb79f, 14, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [14]:
%%sql
-- Moves the extra copies (copy_number > 1) to quarantine, tagged 'duplicate row'.
INSERT INTO orders_quarantine
SELECT order_id, customer_id, customer_email, order_ts, status, channel, sku, product_name,
       category, qty, unit_price, currency, country, fx_reference_date, 'duplicate row'
FROM orders_ranked
WHERE copy_number > 1

StatementMeta(, 225c4ce8-f213-4645-ba3b-a5c55fcbb79f, 15, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [15]:
%%sql
-- Rebuilds orders_clean from the helper TABLE, keeping only the first copy of each row.
-- Reading from orders_ranked (a separate table) avoids the self-reference problem.
CREATE OR REPLACE TABLE orders_clean AS
SELECT order_id, customer_id, customer_email, order_ts, status, channel, sku, product_name,
       category, qty, unit_price, currency, country, fx_reference_date
FROM orders_ranked
WHERE copy_number = 1

StatementMeta(, 225c4ce8-f213-4645-ba3b-a5c55fcbb79f, 16, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [16]:
%%sql
-- Quarantined = 183, remaining = 9085.
SELECT 'quarantined (duplicate row)' AS bucket, COUNT(*) AS rows
FROM orders_quarantine WHERE reject_reason = 'duplicate row'
UNION ALL
SELECT 'remaining in orders_clean', COUNT(*) FROM orders_clean

StatementMeta(, 225c4ce8-f213-4645-ba3b-a5c55fcbb79f, 17, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

## 3. Remove test orders

In [17]:
%%sql
-- Moves 'test' status orders to quarantine, tagged 'test order'.
-- They are internal tests, not real sales.
INSERT INTO orders_quarantine
SELECT *, 'test order' FROM orders_clean WHERE status = 'test'

StatementMeta(, 225c4ce8-f213-4645-ba3b-a5c55fcbb79f, 18, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [18]:
%%sql
-- Removes those test orders from the clean table.
DELETE FROM orders_clean WHERE status = 'test'

StatementMeta(, 225c4ce8-f213-4645-ba3b-a5c55fcbb79f, 19, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [19]:
%%sql
-- Shows what this step quarantined vs what remains in the clean table.
SELECT 'quarantined (test order)' AS bucket, COUNT(*) AS rows
FROM orders_quarantine WHERE reject_reason = 'test order'
UNION ALL
SELECT 'remaining in orders_clean', COUNT(*) FROM orders_clean

StatementMeta(, 225c4ce8-f213-4645-ba3b-a5c55fcbb79f, 20, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

## 4. Remove rows with missing category

In [20]:
%%sql
-- Moves rows with no category to quarantine, tagged 'missing category'.
-- We can't trust a row with no category; fixing it is left for later.
INSERT INTO orders_quarantine
SELECT *, 'missing category' FROM orders_clean WHERE category IS NULL OR TRIM(category) = ''

StatementMeta(, 225c4ce8-f213-4645-ba3b-a5c55fcbb79f, 21, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [21]:
%%sql
-- Removes those missing-category rows from the clean table.
DELETE FROM orders_clean WHERE category IS NULL OR TRIM(category) = ''

StatementMeta(, 225c4ce8-f213-4645-ba3b-a5c55fcbb79f, 22, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [22]:
%%sql
-- Shows what this step quarantined vs what remains.
SELECT 'quarantined (missing category)' AS bucket, COUNT(*) AS rows
FROM orders_quarantine WHERE reject_reason = 'missing category'
UNION ALL
SELECT 'remaining in orders_clean', COUNT(*) FROM orders_clean

StatementMeta(, 225c4ce8-f213-4645-ba3b-a5c55fcbb79f, 23, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

## 5. Remove rows with missing customer id

In [23]:
%%sql
-- Moves rows with no customer id to quarantine, tagged 'missing customer id'.
-- We can't attribute the order to a customer; fixing it is left for later..
INSERT INTO orders_quarantine
SELECT *, 'missing customer id' FROM orders_clean WHERE customer_id IS NULL OR TRIM(customer_id) = ''

StatementMeta(, 225c4ce8-f213-4645-ba3b-a5c55fcbb79f, 24, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [24]:
%%sql
--Removes those missing-customer-id rows from the clean table.
DELETE FROM orders_clean WHERE customer_id IS NULL OR TRIM(customer_id) = ''

StatementMeta(, 225c4ce8-f213-4645-ba3b-a5c55fcbb79f, 25, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [25]:
%%sql
-- Shows what this step quarantined vs what remains.
SELECT 'quarantined (missing customer id)' AS bucket, COUNT(*) AS rows
FROM orders_quarantine WHERE reject_reason = 'missing customer id'
UNION ALL
SELECT 'remaining in orders_clean', COUNT(*) FROM orders_clean

StatementMeta(, 225c4ce8-f213-4645-ba3b-a5c55fcbb79f, 26, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

## 6. Remove invalid quantities

In [26]:
%%sql
-- Moves rows where quantity is 0 or negative to quarantine, tagged 'invalid quantity'.
-- An order can't have zero or negative items. TRY_CAST safely turns the text into a number.
INSERT INTO orders_quarantine
SELECT *, 'invalid quantity' FROM orders_clean WHERE TRY_CAST(qty AS INT) <= 0

StatementMeta(, 225c4ce8-f213-4645-ba3b-a5c55fcbb79f, 27, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [27]:
%%sql
-- Removes those invalid-quantity rows from the clean table.
DELETE FROM orders_clean WHERE TRY_CAST(qty AS INT) <= 0

StatementMeta(, 225c4ce8-f213-4645-ba3b-a5c55fcbb79f, 28, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [28]:
%%sql
-- Shows what this step quarantined vs what remains.
SELECT 'quarantined (invalid quantity)' AS bucket, COUNT(*) AS rows
FROM orders_quarantine WHERE reject_reason = 'invalid quantity'
UNION ALL
SELECT 'remaining in orders_clean', COUNT(*) FROM orders_clean

StatementMeta(, 225c4ce8-f213-4645-ba3b-a5c55fcbb79f, 29, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

## 7. Remove placeholder prices

In [29]:
%%sql
-- Moves rows priced exactly 999999 to quarantine, tagged 'placeholder price'.
-- 999999 is a fake stand-in value, not a real price.
INSERT INTO orders_quarantine
SELECT *, 'placeholder price' FROM orders_clean WHERE TRY_CAST(unit_price AS DOUBLE) = 999999

StatementMeta(, 225c4ce8-f213-4645-ba3b-a5c55fcbb79f, 30, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [30]:
%%sql
-- Removes those placeholder-price rows from the clean table.
DELETE FROM orders_clean WHERE TRY_CAST(unit_price AS DOUBLE) = 999999

StatementMeta(, 225c4ce8-f213-4645-ba3b-a5c55fcbb79f, 31, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [31]:
%%sql
-- Shows what this step quarantined vs what remains.
SELECT 'quarantined (placeholder price)' AS bucket, COUNT(*) AS rows
FROM orders_quarantine WHERE reject_reason = 'placeholder price'
UNION ALL
SELECT 'remaining in orders_clean', COUNT(*) FROM orders_clean

StatementMeta(, 225c4ce8-f213-4645-ba3b-a5c55fcbb79f, 32, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

## 8. Verify — nothing lost / reconcile totals

In [32]:
%%sql
-- Clean + quarantine must equal the original 9268 rows.
-- Two rows — clean and quarantine; they should sum to 9268.
SELECT 'clean' AS bucket, COUNT(*) AS rows FROM orders_clean
UNION ALL
SELECT 'quarantine', COUNT(*) FROM orders_quarantine

StatementMeta(, 225c4ce8-f213-4645-ba3b-a5c55fcbb79f, 33, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

In [33]:
%%sql
-- Full breakdown of why rows were quarantined.
-- duplicate row 183 · invalid quantity 161 · test order 97 · missing customer id 96 · missing category 77 · placeholder price 13 (= 627)- from above

SELECT reject_reason, COUNT(*) AS how_many
FROM orders_quarantine
GROUP BY reject_reason
ORDER BY how_many DESC

StatementMeta(, 225c4ce8-f213-4645-ba3b-a5c55fcbb79f, 34, Finished, Available, Finished, False)

<Spark SQL result set with 6 rows and 2 fields>

## 9. Convert columns to proper types

The clean table currently holds everything as text. Here we give each column its correct type,
based on what it contains and how it's used (do we do math on it? is it a date?).

| Column | Type | Why |
|--------|------|-----|
| order_id | text | an identifier/label — never used for math |
| customer_id | whole number | numeric id used to group orders per customer |
| customer_email | text | it's text |
| order_ts | timestamp | a point in time (arrives in 3 formats, standardised here) |
| status | text | a label |
| channel | text | a label |
| sku | text | a product code — a label, not math |
| product_name | text | text |
| category | text | a label |
| qty | whole number | you count items; used in math (qty × price) |
| unit_price | decimal | money has decimals; used in math |
| line_total | decimal | qty × unit_price (money) |
| currency | text | a code |
| country | text | a code |
| fx_reference_date | date | a calendar date; used to join exchange rates |

Note: ids/codes like order_id and sku stay text on purpose — "looks like a number" is not the
same as "used for math". We build a typed copy first, then swap it in, to avoid reading and
overwriting the same table at once.


In [1]:
%%sql
-- Builds a typed copy of orders_clean (numbers, dates) into a temporary table.
-- A clean table should have real types; building into a separate table avoids the
--       "read and overwrite the same table" problem we hit earlier.
CREATE OR REPLACE TABLE orders_clean_typed AS
SELECT
  order_id,
  CAST(customer_id AS BIGINT)                       AS customer_id,
  customer_email,
  CASE
    WHEN order_ts RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9]{2}:[0-9]{2}:[0-9]{2}$' THEN to_timestamp(order_ts, "yyyy-MM-dd'T'HH:mm:ss")
    WHEN order_ts RLIKE '^[0-9]{9,10}$'                                            THEN timestamp_seconds(CAST(order_ts AS BIGINT))
    WHEN order_ts RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4} [0-9]{2}:[0-9]{2}$'           THEN to_timestamp(order_ts, 'dd/MM/yyyy HH:mm')
  END                                               AS order_ts,
  status,
  channel,
  sku,
  product_name,
  category,
  CAST(qty AS INT)                                  AS qty,
  CAST(unit_price AS DOUBLE)                         AS unit_price,
  ROUND(CAST(qty AS INT) * CAST(unit_price AS DOUBLE), 2) AS line_total,
  currency,
  country,
  CAST(fx_reference_date AS DATE)                    AS fx_reference_date
FROM orders_clean

StatementMeta(, 06cda2a2-11bf-4006-8cff-3dadb61ae5de, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
%%sql
-- Replaces orders_clean with the typed copy.
-- Reads from orders_clean_typed (a different table) so there's no self-reference.
CREATE OR REPLACE TABLE orders_clean AS
SELECT * FROM orders_clean_typed

StatementMeta(, 06cda2a2-11bf-4006-8cff-3dadb61ae5de, 3, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [3]:
%%sql
-- Removes the temporary typed table, no longer needed.
DROP TABLE IF EXISTS orders_clean_typed

StatementMeta(, 06cda2a2-11bf-4006-8cff-3dadb61ae5de, 4, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [4]:
%%sql
-- Shows the column types of orders_clean.
-- qty=int, unit_price/line_total=double, order_ts=timestamp, fx_reference_date=date.
DESCRIBE orders_clean

StatementMeta(, 06cda2a2-11bf-4006-8cff-3dadb61ae5de, 5, Finished, Available, Finished, False)

<Spark SQL result set with 15 rows and 3 fields>

In [5]:
%%sql
-- Confirms the row count didn't change.
SELECT COUNT(*) AS rows FROM orders_clean

StatementMeta(, 06cda2a2-11bf-4006-8cff-3dadb61ae5de, 6, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>